In [ ]:
"""
DSCI 510 Final Project - Step 4A
Build a seed list of 1000 random ISBN-13 values using Open Library subjects.

Output:
  data/raw/isbn_seed_1000.csv
"""

from __future__ import annotations

import os
import random
import time
from typing import Iterable, List, Optional, Set

import pandas as pd
import requests


BASE_URL = "https://openlibrary.org/subjects/{subject}.json"


def safe_get_json(url: str, params: dict, timeout: int = 20, retries: int = 3) -> Optional[dict]:
    """
    Make a GET request and return JSON safely with retries.

    Returns None if it fails.
    """
    headers = {
        "User-Agent": "DSCI510-BookPriceProject/1.0 (educational use)"
    }

    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(url, params=params, headers=headers, timeout=timeout)
            if resp.status_code == 200:
                return resp.json()
            else:
                print(f"[WARN] HTTP {resp.status_code} for {url} params={params}")
        except requests.RequestException as e:
            print(f"[WARN] Request failed (attempt {attempt}/{retries}): {e}")

        time.sleep(1.0 * attempt)  # simple backoff

    return None


def fetch_isbns_for_subject(subject: str, max_pages: int = 12, sleep_s: float = 0.35) -> Set[str]:
    """
    Fetch ISBN-13 values for a given Open Library subject.

    Notes:
    - Open Library subject endpoint returns works, and sometimes embedded edition info.
    - ISBN availability varies by subject and work.
    - We collect only ISBN-13 to keep matching consistent downstream.

    Parameters
    ----------
    subject : str
        Subject slug used by Open Library (e.g., "fantasy")
    max_pages : int
        How many pages to fetch (each ~50 works)
    sleep_s : float
        Delay between requests (polite scraping)

    Returns
    -------
    set[str]
        Unique ISBN-13 values found
    """
    isbns: Set[str] = set()
    url = BASE_URL.format(subject=subject)

    for page in range(max_pages):
        params = {"limit": 50, "offset": page * 50}
        data = safe_get_json(url, params=params)

        if not data:
            print(f"[WARN] No data for subject={subject}, page={page}")
            continue

        works = data.get("works", [])
        if not works:
            break  # no more pages

        # Extract ISBNs (Open Library structure can vary)
        for work in works:
            # Some subjects include "availability" and other metadata but not editions.
            # If "editions" exist, try to pull isbn_13.
            editions = work.get("editions", [])
            for ed in editions:
                # Open Library can store isbn_13 as a list
                for isbn in ed.get("isbn_13", []) or []:
                    if isinstance(isbn, str) and len(isbn) == 13 and isbn.isdigit():
                        isbns.add(isbn)

        time.sleep(sleep_s)

    return isbns


def collect_isbns(subjects: Iterable[str], max_pages_per_subject: int = 12) -> Set[str]:
    """
    Collect ISBN-13 values across many subjects.
    """
    all_isbns: Set[str] = set()

    for subject in subjects:
        print(f"[INFO] Fetching ISBNs for subject: {subject}")
        subject_isbns = fetch_isbns_for_subject(subject, max_pages=max_pages_per_subject)
        print(f"[INFO]   found {len(subject_isbns)} ISBN-13 values")
        all_isbns.update(subject_isbns)

    return all_isbns


def sample_isbns(isbns: Set[str], n: int = 1000, seed: int = 42) -> List[str]:
    """
    Randomly sample n ISBNs from a set, using a fixed seed for reproducibility.
    """
    isbns_list = sorted(list(isbns))
    if len(isbns_list) < n:
        raise ValueError(
            f"Not enough ISBNs collected to sample {n}. Collected={len(isbns_list)}"
        )
    random.seed(seed)
    return random.sample(isbns_list, n)


def save_isbns_to_csv(isbns: List[str], filepath: str) -> None:
    """
    Save ISBN list to CSV with a single column 'isbn'.
    """
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    df = pd.DataFrame({"isbn": isbns})
    df.to_csv(filepath, index=False)


def main() -> None:
    # Subjects you chose (good variety + helps matching)
    subjects = [
        "art",
        "science_fiction",
        "fantasy",
        "biographies",
        "children",
        "cooking",
        "romance",
        "history",
        "mystery_and_detective_stories",
        "medicine",
        "religion",
        "plays",
        "science",
    ]

    print("[INFO] Collecting ISBNs from Open Library...")
    all_isbns = collect_isbns(subjects, max_pages_


In [1]:
"""
DSCI 510 Final Project - Step 4A (Robust)
Build a seed list of 1000 random ISBN-13 values using Open Library.

Key improvement:
- Subjects may have different slugs on Open Library.
- We try multiple slug variants on /subjects/{slug}.json
- If that doesn't work, we fallback to /search.json?subject=<original subject>

Output:
  data/raw/isbn_seed_1000.csv
"""

from __future__ import annotations

import os
import random
import re
import time
from typing import Iterable, List, Optional, Set

import pandas as pd
import requests


SUBJECT_URL = "https://openlibrary.org/subjects/{slug}.json"
SEARCH_URL = "https://openlibrary.org/search.json"


def safe_get_json(url: str, params: dict, timeout: int = 20, retries: int = 3) -> Optional[dict]:
    """GET JSON with simple retries and a polite user-agent."""
    headers = {"User-Agent": "DSCI510-BookPriceProject/1.0 (educational use)"}

    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(url, params=params, headers=headers, timeout=timeout)
            if resp.status_code == 200:
                return resp.json()
            # 404/500 etc.
            # print only minimal noise
            # print(f"[WARN] HTTP {resp.status_code} for {url} params={params}")
        except requests.RequestException as e:
            print(f"[WARN] Request failed (attempt {attempt}/{retries}): {e}")

        time.sleep(0.75 * attempt)

    return None


def normalize_isbn13(values: Iterable[str]) -> Set[str]:
    """Return only clean ISBN-13 digits."""
    out: Set[str] = set()
    for v in values:
        if not isinstance(v, str):
            continue
        v = v.strip()
        if len(v) == 13 and v.isdigit():
            out.add(v)
    return out


def subject_slug_variants(subject: str) -> List[str]:
    """
    Create multiple possible Open Library subject slugs from a human subject string.
    Example: "Mystery and Detective Stories" ->
      ["mystery_and_detective_stories", "mystery-and-detective-stories", ...]
    """
    s = subject.strip().lower()

    # common normalization
    s = s.replace("&", "and")
    s = re.sub(r"[^\w\s-]", "", s)      # remove punctuation except underscore/hyphen
    s_space = re.sub(r"\s+", " ", s).strip()

    underscore = s_space.replace(" ", "_")
    hyphen = s_space.replace(" ", "-")
    compact = s_space.replace(" ", "")

    # Sometimes OpenLibrary subjects omit "and"
    no_and_underscore = underscore.replace("_and_", "_")
    no_and_hyphen = hyphen.replace("-and-", "-")

    variants = []
    for v in [underscore, hyphen, no_and_underscore, no_and_hyphen, compact]:
        if v and v not in variants:
            variants.append(v)

    return variants


def fetch_isbns_via_subject_endpoint(slug: str, max_pages: int = 12, sleep_s: float = 0.35) -> Set[str]:
    """
    Try to collect ISBN-13s from the subject endpoint.
    Note: subject endpoint does NOT always include edition ISBNs, so this may return 0.
    """
    isbns: Set[str] = set()
    url = SUBJECT_URL.format(slug=slug)

    for page in range(max_pages):
        params = {"limit": 50, "offset": page * 50}
        data = safe_get_json(url, params=params)
        if not data:
            # slug might be invalid
            break

        works = data.get("works", [])
        if not works:
            break

        # The subject endpoint structure varies.
        # Sometimes it contains "availability"/work metadata but not editions.
        # We'll try any embedded isbn fields if present (rare), otherwise this yields 0.
        for work in works:
            # Occasionally there can be 'isbn' / 'isbn_13' fields in works (not common)
            if "isbn_13" in work and isinstance(work["isbn_13"], list):
                isbns |= normalize_isbn13(work["isbn_13"])
            if "isbn" in work and isinstance(work["isbn"], list):
                isbns |= normalize_isbn13(work["isbn"])

            # If editions are embedded (sometimes), use them
            editions = work.get("editions", [])
            for ed in editions:
                if "isbn_13" in ed and isinstance(ed["isbn_13"], list):
                    isbns |= normalize_isbn13(ed["isbn_13"])
                if "isbn" in ed and isinstance(ed["isbn"], list):
                    isbns |= normalize_isbn13(ed["isbn"])

        time.sleep(sleep_s)

    return isbns


def fetch_isbns_via_search_endpoint(subject_query: str, max_pages: int = 40, sleep_s: float = 0.25) -> Set[str]:
    """
    Collect ISBN-13s using the search endpoint:
      /search.json?subject=<subject_query>&limit=100&page=1
    This is more flexible than the subjects endpoint and usually returns 'isbn' arrays in docs.
    """
    isbns: Set[str] = set()

    for page in range(1, max_pages + 1):
        params = {
            "subject": subject_query,
            "limit": 100,
            "page": page,
            # Ask for only what we need (lighter responses)
            "fields": "isbn,title,author_name,key",
        }
        data = safe_get_json(SEARCH_URL, params=params)
        if not data:
            break

        docs = data.get("docs", [])
        if not docs:
            break

        for d in docs:
            # 'isbn' often contains ISBN-10 and ISBN-13 mixed
            isbn_list = d.get("isbn", [])
            if isinstance(isbn_list, list):
                isbns |= normalize_isbn13(isbn_list)

        time.sleep(sleep_s)

    return isbns


def fetch_isbns_for_subject(subject: str) -> Set[str]:
    """
    Robust subject ISBN collection:
    1) Try multiple subject slugs on /subjects/{slug}.json
    2) Fallback to /search.json?subject=<subject> if needed
    """
    collected: Set[str] = set()

    # 1) Try slugs
    variants = subject_slug_variants(subject)
    best_slug = None
    best_count = 0

    for slug in variants:
        isbns = fetch_isbns_via_subject_endpoint(slug)
        if len(isbns) > best_count:
            best_count = len(isbns)
            best_slug = slug
            collected = isbns

        # If we already got a decent amount, stop early
        if best_count >= 250:
            break

    if best_slug and best_count > 0:
        print(f"[INFO]   subject endpoint worked: slug='{best_slug}' -> {best_count} ISBN-13s")
        return collected

    # 2) Fallback: search endpoint
    isbns_search = fetch_isbns_via_search_endpoint(subject)
    print(f"[INFO]   fallback search endpoint: subject='{subject}' -> {len(isbns_search)} ISBN-13s")
    return isbns_search


def collect_isbns(subjects: Iterable[str]) -> Set[str]:
    """Collect ISBNs across multiple human-readable subjects."""
    all_isbns: Set[str] = set()

    for subject in subjects:
        print(f"[INFO] Fetching ISBNs for: {subject}")
        isbns = fetch_isbns_for_subject(subject)
        print(f"[INFO]   total captured for '{subject}': {len(isbns)}")
        all_isbns.update(isbns)

    return all_isbns


def sample_isbns(isbns: Set[str], n: int = 1000, seed: int = 42) -> List[str]:
    """Random sample with reproducibility."""
    isbns_list = sorted(isbns)
    if len(isbns_list) < n:
        raise ValueError(f"Not enough ISBN-13s collected to sample {n}. Collected={len(isbns_list)}")
    random.seed(seed)
    return random.sample(isbns_list, n)


def save_isbns_to_csv(isbns: List[str], filepath: str) -> None:
    """Save to CSV in data/raw."""
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    pd.DataFrame({"isbn": isbns}).to_csv(filepath, index=False)


def main() -> None:
    # Use human-friendly subjects (code will handle mismatched slugs)
    subjects = [
        "Art",
        "Science Fiction",
        "Fantasy",
        "Biographies",
        "Children",
        "Recipes",
        "Romance",
        "History",
        "Mystery and Detective Stories",
        "Medicine",
        "Religion",
        "Plays",
        "Science",
    ]

    print("[INFO] Collecting ISBNs from Open Library (robust mode)...")
    all_isbns = collect_isbns(subjects)
    print(f"[INFO] Total unique ISBN-13 collected: {len(all_isbns)}")

    sampled = sample_isbns(all_isbns, n=1000, seed=42)
    output_path = "data/raw/isbn_seed_1000.csv"
    save_isbns_to_csv(sampled, output_path)

    print(f"[DONE] Saved {len(sampled)} ISBNs to: {output_path}")


if __name__ == "__main__":
    main()


[INFO] Collecting ISBNs from Open Library (robust mode)...
[INFO] Fetching ISBNs for: Art
[INFO]   fallback search endpoint: subject='Art' -> 58387 ISBN-13s
[INFO]   total captured for 'Art': 58387
[INFO] Fetching ISBNs for: Science Fiction
[INFO]   fallback search endpoint: subject='Science Fiction' -> 96568 ISBN-13s
[INFO]   total captured for 'Science Fiction': 96568
[INFO] Fetching ISBNs for: Fantasy
[INFO]   fallback search endpoint: subject='Fantasy' -> 137543 ISBN-13s
[INFO]   total captured for 'Fantasy': 137543
[INFO] Fetching ISBNs for: Biographies
[INFO]   fallback search endpoint: subject='Biographies' -> 76684 ISBN-13s
[INFO]   total captured for 'Biographies': 76684
[INFO] Fetching ISBNs for: Children
[INFO]   fallback search endpoint: subject='Children' -> 218520 ISBN-13s
[INFO]   total captured for 'Children': 218520
[INFO] Fetching ISBNs for: Recipes
[INFO]   fallback search endpoint: subject='Recipes' -> 6002 ISBN-13s
[INFO]   total captured for 'Recipes': 6002
[INFO]